In [ ]:
import csv

def check_winner(board):
    win_configs = [
        [0, 1, 2], [3, 4, 5], [6, 7, 8], # Lignes
        [0, 3, 6], [1, 4, 7], [2, 5, 8], # Colonnes
        [0, 4, 8], [2, 4, 6]             # Diagonales
    ]
    for config in win_configs:
        if board[config[0]] == board[config[1]] == board[config[2]] != 0:
            return board[config[0]] # 1 pour X, -1 pour O
    if 0 not in board:
        return 0 # Match nul
    return None # Partie en cours

def minimax(board, depth, is_maximizing, alpha, beta):
    result = check_winner(board)
    if result is not None:
        return result

    if is_maximizing:
        best_score = -float('inf')
        for i in range(9):
            if board[i] == 0:
                board[i] = 1
                score = minimax(board, depth + 1, False, alpha, beta)
                board[i] = 0
                best_score = max(score, best_score)
                alpha = max(alpha, score)
                if beta <= alpha: break
        return best_score
    else:
        best_score = float('inf')
        for i in range(9):
            if board[i] == 0:
                board[i] = -1
                score = minimax(board, depth + 1, True, alpha, beta)
                board[i] = 0
                best_score = min(score, best_score)
                beta = min(beta, score)
                if beta <= alpha: break
        return best_score

def generate_dataset():
    dataset = []
    seen_states = set()

    def solve(board, turn_of_x):
        state_tuple = tuple(board)
        if state_tuple in seen_states:
            return
        seen_states.add(state_tuple)

        # Si c'est au tour de X, on évalue et on enregistre
        if turn_of_x:
            res = check_winner(board)
            # On n'enregistre que si la partie n'est pas déjà finie
            if res is None:
                score = minimax(list(board), 0, True, -float('inf'), float('inf'))

                # Encodage 18 bits
                row = []
                for val in board: row.append(1 if val == 1 else 0) # c0_x...c8_x
                for val in board: row.append(1 if val == -1 else 0) # c0_o...c8_o

                # Cibles
                row.append(1 if score == 1 else 0)  # x_wins
                row.append(1 if score == 0 else 0)  # is_draw
                dataset.append(row)

        # Exploration des coups suivants
        res = check_winner(board)
        if res is None:
            for i in range(9):
                if board[i] == 0:
                    board[i] = 1 if turn_of_x else -1
                    solve(board, not turn_of_x)
                    board[i] = 0

    solve([0]*9, True)
    return dataset

# Sauvegarde en CSV
data = generate_dataset()
headers = [f'c{i}_x' for i in range(9)] + [f'c{i}_o' for i in range(9)] + ['x_wins', 'is_draw']

with open('tic_tac_toe_dataset.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(headers)
    writer.writerows(data)

print(f"Dataset généré avec {len(data)} états uniques pour X.")